# درس ۳ — مهندسی ویژگی‌ها (Feature Engineering)
## پروژه تخمین هزینه حمل Atramart

در درس قبل `X` و `y` را جدا کردیم و داده را به آموزش و آزمون تقسیم کردیم. حالا باید داده‌ها را به شکلی تبدیل کنیم که الگوریتم‌های یادگیری ماشین بتوانند با آن‌ها محاسبه انجام دهند.

**مهندسی ویژگی** یعنی آماده‌سازی یا ساختن اطلاعات ورودی مفید برای مدل. در این درس سه کار انجام می‌دهیم:
1. از تاریخ، ویژگی‌های عددی استخراج می‌کنیم.
2. مقدار خالی یک ستون دسته‌ای را مدیریت می‌کنیم.
3. متن‌های دسته‌ای را به ستون‌های عددی تبدیل می‌کنیم.

> دیتاست فعلی مصنوعی و آموزشی است؛ فعلاً روش را تمرین می‌کنیم.

## ۱. کتابخانه‌ها و آماده‌کردن X و y

این نوت‌بوک مستقل است؛ بنابراین فایل را دوباره می‌خوانیم و تقسیم آموزش/آزمون را با همان تنظیمات درس قبل می‌سازیم. `random_state=42` کمک می‌کند همان تقسیم تکرار شود.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = Path("../data/raw/atramart_random_shipping_dataset.csv")
df = pd.read_csv(DATA_PATH, parse_dates=["shipping_date"])

TARGET = "real_cost"
FEATURES = [
    "weight", "items_count", "base_city", "destination_city",
    "destination_region", "product_category", "shipping_date",
    "package_count", "carrier_name", "service_type"
]

X = df[FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

## ۲. استخراج ویژگی از تاریخ

ستون `shipping_date` از نوع تاریخ است. بسیاری از مدل‌ها مستقیماً با تاریخ خام کار نمی‌کنند؛ بنابراین از آن اطلاعات ساده‌تری استخراج می‌کنیم:

- `shipping_month`: ماه ارسال (۱ تا ۱۲)
- `shipping_dayofweek`: روز هفته (دوشنبه=۰ تا یکشنبه=۶ در pandas)
- `shipping_is_weekend`: آیا روز ارسال جمعه/شنبه است؟ (در داده‌های میلادی، جمعه=۴ و شنبه=۵)

**نکته:** در این تمرین آخرهفته ایران را جمعه و شنبه در نظر می‌گیریم. pandas روز هفته را با تقویم میلادی شماره‌گذاری می‌کند؛ برای تاریخ‌های شمسی/تعطیلات رسمی باید بعداً تقویم دقیق ایران را اضافه کنیم.

In [ ]:
for data in (X_train, X_test):
    data["shipping_month"] = data["shipping_date"].dt.month
    data["shipping_dayofweek"] = data["shipping_date"].dt.dayofweek
    data["shipping_is_weekend"] = data["shipping_dayofweek"].isin([4, 5]).astype(int)

# تاریخ خام را بعد از استخراج ویژگی‌ها حذف می‌کنیم
X_train = X_train.drop(columns=["shipping_date"])
X_test = X_test.drop(columns=["shipping_date"])

X_train[["shipping_month", "shipping_dayofweek", "shipping_is_weekend"]].head()

## ۳. مدیریت مقدارهای خالی در destination_region

برای سفارش‌های خارج از تهران، ممکن است `destination_region` خالی باشد؛ چون منطقه/محله فقط برای تحویل‌های داخل تهران معنا دارد.

در این تمرین، مقدار خالی را با متن `ثبت نشده` جایگزین می‌کنیم. این به معنی حدس زدن منطقه نیست؛ فقط یک برچسب مشخص برای «اطلاعات موجود نیست» می‌سازیم.

In [ ]:
X_train["destination_region"] = X_train["destination_region"].fillna("ثبت نشده")
X_test["destination_region"] = X_test["destination_region"].fillna("ثبت نشده")

print("مقادیر خالی train:", X_train.isna().sum().sum())
print("مقادیر خالی test :", X_test.isna().sum().sum())

## ۴. تبدیل متن دسته‌ای به عدد: One-Hot Encoding

ستون‌هایی مثل `carrier_name` متن هستند: Post، Tipax، Cargo و Courier. مدل‌های معمولی نمی‌توانند این متن‌ها را مستقیماً در محاسبات عددی استفاده کنند.

یک روش ساده **One-Hot Encoding** است: برای هر دسته یک ستون صفر/یک می‌سازیم. مثلاً ستون `carrier_name_Post` اگر شرکت پست باشد ۱ و در غیر این صورت ۰ می‌شود.

از `get_dummies` استفاده می‌کنیم. آن را فقط روی داده آموزش اجرا می‌کنیم، سپس ستون‌های آزمون را با ستون‌های آموزش هماهنگ می‌کنیم. این کار باعث می‌شود ساختار ورودی آزمون دقیقاً با آموزش یکی باشد.

In [ ]:
categorical_columns = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
print("ستون‌های دسته‌ای:", categorical_columns)

X_train_encoded = pd.get_dummies(
    X_train, columns=categorical_columns, dtype=int
)
X_test_encoded = pd.get_dummies(
    X_test, columns=categorical_columns, dtype=int
)

# ستون‌های test را با ستون‌های train یکسان می‌کنیم
X_test_encoded = X_test_encoded.reindex(
    columns=X_train_encoded.columns, fill_value=0
)

print("شکل X_train_encoded:", X_train_encoded.shape)
print("شکل X_test_encoded :", X_test_encoded.shape)

## ۵. بررسی نتیجه

حالا ورودی‌های آماده‌شده باید عددی باشند و تعداد ستون‌های train و test برابر باشد. `select_dtypes` کمک می‌کند ستون‌هایی را پیدا کنیم که هنوز نوع متنی دارند.

In [ ]:
print("ستون‌های غیرعددی در train:")
print(X_train_encoded.select_dtypes(exclude="number").columns.tolist())

print("آیا ستون‌های train و test کاملاً یکسان‌اند؟")
print(X_train_encoded.columns.equals(X_test_encoded.columns))

X_train_encoded.head()

## ۶. تمرین پایان درس

۱. چرا `shipping_date` خام را حذف کردیم و به‌جایش ماه و روز هفته ساختیم؟
۲. One-Hot Encoding چه مشکلی را حل می‌کند؟
۳. چرا ستون‌های `X_test_encoded` را با ستون‌های `X_train_encoded` هماهنگ کردیم؟
۴. خروجی دو بررسی آخر چه چیزی را درباره آماده بودن داده‌ها نشان می‌دهد؟

**توقف اینجا:** هنوز مدل را آموزش نمی‌دهیم. ابتدا این درس را اجرا کن و تمرین‌ها را پاسخ بده؛ سپس به سراغ ساخت اولین مدل می‌رویم.